# Calcusol · R2 · Evaluación del modelo (genera el Excel que lee R3)

Este cuaderno ejecuta **`src/calcusol/validacion/evaluar_modelo.py`** en una máquina con más RAM y núcleos (por ejemplo Google Colab). El script es el mismo del repositorio: usa las mismas 17 variables, la misma partición y el mismo modelo que el entrenamiento. Produce `evaluacion_lightgbm.xlsx`, con las hojas `R3_casos`, `R3_mejora_por_sitio` y `R3_serie_horaria` (las que lee R3) y las de evaluación (`resumen_modelos`, `intervalos_confianza`, `cv_resumen`, `cv_no_fuga`).

**Estructura que necesitas en Google Drive** (una carpeta, por ejemplo `MyDrive/calcusol/`, que replica el repositorio):
```
calcusol/
  src/calcusol/...                          (copia de la carpeta src/ del repositorio)
  data/gold/dataset_modelo.parquet
  models/radiacion/v1/modelo_lgbm.txt
  models/radiacion/v1/metadatos.json
```
**GPU:** no ayuda aquí. Todo corre en CPU (LightGBM). Lo que acelera es tener más núcleos y RAM (por ejemplo, el entorno de RAM alta de Colab).

**Cuánto tarda:** el modo `completo` hace 10 validaciones cruzadas (por Sitio y por celda NASA), la curva de aprendizaje y el A/B de preliminares; en un PC modesto se estimó entre 2 y 3 horas. El modo `rapido` omite las partes opcionales y es bastante más corto (estimado, sin medir). Con menos núcleos, tarda más.


In [ ]:
# Paso 0 · Instalar dependencias
!pip install -q pvlib pyarrow scikit-learn scipy lightgbm openpyxl pyyaml

In [ ]:
# Paso 1 · Parámetros
CARPETA_DRIVE = '/content/drive/MyDrive/calcusol'   # solo se usa en Colab

# 'completo' = todo lo que hace el script (Excel completo, el que debe quedar en el repositorio)
# 'rapido'   = solo la validación cruzada por Sitio con ubicación (lo que necesita R3);
#              el Excel queda SIN: esquema por celda NASA, curva de aprendizaje, A/B de preliminares y modelo sin ubicación
MODO = 'completo'

PRESETS = {
    'completo': {},
    'rapido': {'CV_ESQUEMA_CELDA': 'false', 'CV_CURVA': 'false', 'HACER_AB_PRELIMINARES': 'false', 'CV_CON_SIN_UBICACION': 'false'},
}
assert MODO in PRESETS, 'MODO debe ser completo o rapido'

In [ ]:
# Paso 2 · Rutas y verificación de archivos
import os, sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path(CARPETA_DRIVE)
else:
    RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'calcusol').is_dir())
sys.path.insert(0, str(RAIZ / 'src'))

SCRIPT  = RAIZ / 'src' / 'calcusol' / 'validacion' / 'evaluar_modelo.py'
SALIDA  = RAIZ / 'src' / 'calcusol' / 'validacion' / 'resultados' / 'evaluacion_lightgbm.xlsx'
for f in [SCRIPT, RAIZ / 'data' / 'gold' / 'dataset_modelo.parquet', RAIZ / 'models' / 'radiacion' / 'v1' / 'modelo_lgbm.txt']:
    print('OK    ' if f.exists() else 'FALTA ', f)
    assert f.exists(), f'Falta {f}: revisa la estructura de carpetas del Paso 0'

for k, v in PRESETS[MODO].items():            # opciones del script, sin editarlo
    os.environ['EVALUAR_' + k] = v
print('Modo:', MODO, '| opciones cambiadas:', PRESETS[MODO] or 'ninguna')
print('El Excel se escribirá en:', SALIDA)

## Paso 3 · Ejecutar la evaluación
Imprime cada etapa con su tiempo. Si Colab se desconecta, hay que repetir: el Excel solo se escribe al final.

In [ ]:
import runpy
runpy.run_path(str(SCRIPT), run_name='__main__')

## Paso 4 · Comprobar el resultado
Revisa que el Excel tenga las hojas que lee R3 y que R3 pueda leer los 3 casos.

In [ ]:
import pandas as pd
print(pd.ExcelFile(SALIDA).sheet_names)

from calcusol.fisica_finanzas import calcusol_r3 as r3
cfg = r3.cargar_config()
cfg['datos']['casos_r2'] = str(SALIDA)
serie = r3.cargar_serie_r2(cfg)
casos = r3.cargar_casos_r2(cfg, serie)
display(casos[['Caso', 'CodigoEstacion', 'NombreEstacion', 'Departamento', 'Mejora_pct']].round(1))

In [ ]:
# Cifras principales (examen espacial = lugares nuevos, la cifra defendible)
res = pd.read_excel(SALIDA, sheet_name='resumen_modelos', header=2)
display(res[res['examen'].str.startswith('Espacial')][['modelo', 'estaciones', 'MAE_hora', 'RMSE_hora', 'MBE_hora', 'sesgo_pct']].round(2))

## Paso 5 · Llevar el Excel al repositorio
Descarga `evaluacion_lightgbm.xlsx` (está en `calcusol/src/calcusol/validacion/resultados/` de tu Drive) y cópialo en esa misma ruta del repositorio. Con eso R3 y el notebook de física y finanzas usan los datos nuevos. Si corriste en modo `rapido`, **no sobrescribas** el Excel completo: guárdalo con otro nombre.